# Accuracy deterioration across dated labeled batches

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/deterioration.ipynb)

Companion notebook for the ReliableML paper `docs/latex/deterioration.tex`. It uses `assess_deterioration` (also `ReliablePipeline.assess_deterioration`): per-batch accuracy, an OLS slope on time, a within-batch bootstrap interval and a conservative flag.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Method

For labeled batches $j=1,\dots,J$ ordered by timestamp $t_j$ with $n_j$ rows, accuracy is $a_j=\frac1{n_j}\sum_i\mathbf 1\{\hat y_{ji}=y_{ji}\}$. Ordinary least squares fits (each batch weighted equally)
$$a_j=\beta_0+\beta_1t_j+\varepsilon_j,\qquad \hat\beta_1=\frac{\sum_j(t_j-\bar t)(a_j-\bar a)}{\sum_j(t_j-\bar t)^2}.$$
ISO date strings are converted to days, so $\beta_1$ is "accuracy change per day"; numeric timestamps keep their own unit. At least three distinct timestamps are required, and numeric and ISO timestamps may not be mixed.

**Within-batch bootstrap.** For replicate $b=1,\dots,B$ resample $n_j$ rows *with replacement inside every batch*, recompute $a^{(b)}_j$, refit and store $\hat\beta_1^{(b)}$. The percentile interval at confidence $1-\alpha$ is
$$\bigl[\,q_{\alpha/2}(\hat\beta_1^{(b)}),\;q_{1-\alpha/2}(\hat\beta_1^{(b)})\,\bigr].$$
The conservative flag is $\texttt{deterioration\_detected}=\mathbf 1\{q_{1-\alpha/2}<0\}$: the whole interval must be below zero. The regression $p$-value is the ordinary slope test and is reported separately; it ignores within-batch sampling error. This is a descriptive first-order trend, not a change-point or causal test.

## 2. Synthetic dated batches: drifting model vs stable control

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from ml_reliability import assess_deterioration

rng = np.random.default_rng(11)
dates = pd.date_range("2025-01-01", periods=10, freq="7D").strftime("%Y-%m-%d").tolist()

def make_history(accuracy_by_batch, n=200):
    history = []
    for date, acc in zip(dates, accuracy_by_batch):
        y_true = rng.integers(0, 2, n)
        correct = rng.random(n) < acc
        history.append({"timestamp": date, "y_true": y_true, "y_pred": np.where(correct, y_true, 1 - y_true)})
    return history

decline = np.linspace(0.92, 0.80, len(dates))  # generating accuracy falls 0.12 over 63 days
stable = np.full(len(dates), 0.88)
reports = {
    "declining": assess_deterioration(make_history(decline), bootstrap_samples=1000, random_seed=0),
    "stable": assess_deterioration(make_history(stable), bootstrap_samples=1000, random_seed=0),
}
for name, rep in reports.items():
    low, high = rep.slope_confidence_interval
    print(f"{name:10s} slope/day={rep.accuracy_slope:+.5f}  95% CI=[{low:+.5f}, {high:+.5f}]  "
          f"p={rep.p_value:.4f}  deterioration_detected={rep.deterioration_detected}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (name, rep) in zip(axes, reports.items()):
    days = np.array([(pd.Timestamp(o["timestamp"]) - pd.Timestamp(dates[0])).days for o in rep.observations])
    acc = np.array([float(o["accuracy"]) for o in rep.observations])
    fit = stats.linregress(days, acc)
    ax.plot(days, acc, "o", label="batch accuracy")
    ax.plot(days, fit.intercept + fit.slope * days, "-", label="OLS fit")
    ax.set_title(f"{name}: slope {rep.accuracy_slope:+.4f}/day, CI {tuple(round(v, 4) for v in rep.slope_confidence_interval)}", fontsize=9)
    ax.set_xlabel("days since first batch"); ax.legend()
axes[0].set_ylabel("accuracy"); plt.tight_layout(); plt.show()

### Verify the slope and p-value against scipy

In [ ]:
rep = reports["declining"]
days = np.array([(pd.Timestamp(o["timestamp"]) - pd.Timestamp(dates[0])).days for o in rep.observations])
acc = np.array([float(o["accuracy"]) for o in rep.observations])
fit = stats.linregress(days, acc)
print("slope  report:", round(rep.accuracy_slope, 8), " scipy:", round(fit.slope, 8))
print("p      report:", round(rep.p_value, 8), " scipy:", round(fit.pvalue, 8))
print(pd.DataFrame([{"timestamp": o["timestamp"], "n": o["sample_count"], "accuracy": float(o["accuracy"])} for o in rep.observations]).round(3).to_string(index=False))

### Input requirements

Numeric timestamps work too (slope per unit). Fewer than three distinct dates, or mixed timestamp types, raise a clear error rather than returning a meaningless slope.

In [ ]:
numeric = [{"timestamp": i, "y_true": h["y_true"], "y_pred": h["y_pred"]} for i, h in enumerate(make_history(decline))]
num_rep = assess_deterioration(numeric, bootstrap_samples=500, random_seed=0)
print("numeric time: slope per unit =", round(num_rep.accuracy_slope, 5))

for label, bad in {"two dates only": make_history(decline)[:2],
                   "mixed timestamp types": [dict(make_history(decline)[0], timestamp=0)] + make_history(decline)[1:]}.items():
    try:
        assess_deterioration(bad, bootstrap_samples=100)
    except Exception as error:
        print(f"{label}: {type(error).__name__}: {error}")

## 3. Real data: breast cancer ordered into pseudo-batches

Breast cancer has no timestamps. To exercise the API on a bundled dataset we sort the test rows by a feature and cut them into ordered pseudo-batches. **This is not real time**: it shows how performance changes along that feature, and the output must not be read as temporal deterioration.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

bc = load_breast_cancer(as_frame=True)
Xb = bc.data[["mean radius", "mean texture", "mean smoothness", "mean compactness", "mean symmetry"]]
Xtr, Xte, ytr, yte = train_test_split(Xb, bc.target, test_size=0.5, random_state=0, stratify=bc.target)
model = RandomForestClassifier(100, random_state=0).fit(Xtr, ytr)
order = np.argsort(Xte["mean texture"].to_numpy())
pseudo = []
for k, idx in enumerate(np.array_split(order, 6)):
    rows = Xte.iloc[idx]
    pseudo.append({"timestamp": k, "y_true": yte.iloc[idx].to_numpy(), "y_pred": model.predict(rows)})
bc_rep = assess_deterioration(pseudo, bootstrap_samples=1000, random_seed=0)
print("pseudo-batch accuracies:", [round(float(o["accuracy"]), 3) for o in bc_rep.observations], "| n per batch:", [o["sample_count"] for o in bc_rep.observations])
print("slope", round(bc_rep.accuracy_slope, 4), "CI", tuple(round(v, 4) for v in bc_rep.slope_confidence_interval),
      "p", round(bc_rep.p_value, 3), "flag", bc_rep.deterioration_detected)

## Interpretation and limitations

**How to read the output.** The slope is the estimated accuracy change per unit time; the interval is the plausible range given within-batch sampling noise, and the flag fires only if the entire interval is negative. The stable control shows what 'no detected deterioration' looks like with the same batch sizes, and the pseudo-batch example shows that a slope can be produced by any ordering, which is why the ordering must be real time to be interpreted as deterioration.

**Limitations.** Accuracy needs delayed labels; the linear trend can miss abrupt breaks, seasonality or recovery; batches are weighted equally regardless of size; the bootstrap assumes independent rows within a batch (autocorrelated or repeated patients violate this); the regression p-value ignores within-batch error and is shown only for reference; a declining trend may reflect case-mix, policy or label changes rather than model decay; and with few batches the power to detect a decline is low.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.